In [1]:
from dataclasses import dataclass
from enum import Enum
import random
import time
import uuid
from typing import Any, Callable, Dict, List, Optional


# ==========================================
# 1. 归一化异常体系 (ErrorKind & ToolError & Normalizer)
# ==========================================
class ErrorKind(Enum):
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"
    RATE_LIMITED = "rate_limited"
    TIMEOUT = "timeout"
    OVERLOADED = "overloaded"
    CANCELLED = "cancelled"
    INTERNAL = "internal"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool
    status_code: Optional[int] = None
    retry_after: Optional[float] = None


def normalize_error(
    status_code: Optional[int] = None,
    exc: Optional[Exception] = None,
    retry_after: Optional[float] = None,
    is_bulkhead_rejected: bool = False,
) -> ToolError:
    """将 Raw Error 归一化映射为标准 ToolError"""
    # 1. 系统过载 / 隔舱拒绝
    if is_bulkhead_rejected:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="Bulkhead capacity full",
            retryable=False,  # 避免对已过载的系统发起风暴重试
            status_code=503,
        )

    # 2. Python 原生 Exception 映射
    if exc is not None:
        if isinstance(exc, TimeoutError):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=str(exc) or "Request execution timed out",
                retryable=True,
            )
        # 兜底 Python 原生未知/崩溃异常
        return ToolError(
            kind=ErrorKind.INTERNAL,
            message=f"Internal exception: {type(exc).__name__} - {str(exc)}",
            retryable=False,
        )

    # 3. HTTP Status Code 映射
    if status_code is not None:
        if status_code == 429:
            return ToolError(
                kind=ErrorKind.RATE_LIMITED,
                message="HTTP 429 Too Many Requests",
                retryable=True,
                status_code=429,
                retry_after=retry_after or 0.1,  # 优先采用服务器建议的 Retry-After
            )
        if status_code in (500, 502, 503, 504):
            return ToolError(
                kind=ErrorKind.RETRYABLE,
                message=f"HTTP {status_code} Server Error",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (408,):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=f"HTTP {status_code} Request Timeout",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (400, 401, 403, 404, 422):
            return ToolError(
                kind=ErrorKind.NON_RETRYABLE,
                message=f"HTTP {status_code} Client Error",
                retryable=False,
                status_code=status_code,
            )

    # 4. 无法归类的未知错误
    return ToolError(
        kind=ErrorKind.INTERNAL,
        message=f"Unknown raw error (status={status_code})",
        retryable=False,
        status_code=status_code,
    )


# ==========================================
# 2. 基础组件与配置 (与 Day 26 保持一致)
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    idempotency_required: bool
    breaker: CircuitBreaker
    bulkhead: Bulkhead


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


# ==========================================
# 3. 基于 Error Normalization 的 ToolRuntime
# ==========================================
class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> Dict[str, Any]:
        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        if config.idempotency_required and "idempotency_key" not in args:
            args["idempotency_key"] = f"idempotent-{uuid.uuid4().hex[:8]}"

        if deadline is None:
            deadline = time.time() + 10.0

        # Breaker Gate
        if not config.breaker.can_call():
            return {
                "status": "FAST_FAIL",
                "result": None,
                "error": f"CircuitBreaker for {tool_name} is OPEN",
            }

        retry_count = 0

        while True:
            # Bulkhead Acquire
            if not config.bulkhead.try_acquire():
                # 归一化过载异常
                tool_error = normalize_error(is_bulkhead_rejected=True)
                return {
                    "status": "BULKHEAD_REJECTED",
                    "error_kind": tool_error.kind.value,
                    "error": tool_error.message,
                }

            raw_result = None
            caught_exc = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                caught_exc = e
            finally:
                config.bulkhead.release()

            # ----------------------------------------------------
            # 核心改进：Raw Error -> Normalizer -> ToolError
            # ----------------------------------------------------
            status_code = None
            retry_after = None
            if isinstance(raw_result, dict):
                status_code = raw_result.get("status")
                retry_after = raw_result.get("retry_after")

            # 只有当非 200 状态码或捕获异常时才触发归一化
            if caught_exc is not None or (status_code and status_code != 200):
                tool_error = normalize_error(
                    status_code=status_code,
                    exc=caught_exc,
                    retry_after=retry_after,
                )
            else:
                tool_error = None

            # 1. 成功处理路径
            if tool_error is None:
                config.breaker.record_success()
                return {
                    "status": "SUCCESS",
                    "result": raw_result,
                    "attempts": retry_count + 1,
                    "idempotency_key": args.get("idempotency_key"),
                }

            # 2. 失败处理路径：打卡记录给 Breaker
            config.breaker.record_failure()

            # ----------------------------------------------------
            # Policy 基于表达清晰的 ErrorKind 做出策略分支判断
            # ----------------------------------------------------
            if not tool_error.retryable:
                return {
                    "status": "FATAL_ERROR",
                    "error_kind": tool_error.kind.value,
                    "error": tool_error.message,
                    "attempts": retry_count + 1,
                }

            # 检查重试上限
            if retry_count >= config.max_retries:
                return {
                    "status": "MAX_RETRIES_EXCEEDED",
                    "error_kind": tool_error.kind.value,
                    "error": f"Reached max retries ({config.max_retries}). Last: {tool_error.message}",
                    "attempts": retry_count + 1,
                }

            # 计算 Backoff（若有 RATE_LIMITED 指定的 retry_after，优先结合使用）
            if tool_error.kind == ErrorKind.RATE_LIMITED and tool_error.retry_after:
                backoff = tool_error.retry_after
            else:
                backoff = config.base_delay * (2 ** retry_count) + random.uniform(0.0, 0.01)

            now = time.time()
            if (deadline - now) < (backoff + config.timeout):
                return {
                    "status": "DEADLINE_EXCEEDED",
                    "error_kind": ErrorKind.TIMEOUT.value,
                    "error": "Deadline exceeded before next retry backoff",
                }

            time.sleep(backoff)
            retry_count += 1


# ==========================================
# 4. Mock 工具（配合预设错误响应）
# ==========================================
class SequenceMockTool:
    def __init__(self, sequence: List[Any]):
        self.sequence = sequence

    def __call__(self, args: Dict[str, Any]) -> Dict[str, Any]:
        item = self.sequence.pop(0) if self.sequence else 200
        if isinstance(item, Exception):
            raise item
        if isinstance(item, dict):
            return item
        return {"status": item, "data": f"Response status {item}"}


# ==========================================
# 5. 4 个核心 Scenario 验证
# ==========================================
if __name__ == "__main__":
    registry = ToolRegistry()

    def make_config(name: str, max_retries: int = 1) -> ToolConfig:
        return ToolConfig(
            name=name,
            max_retries=max_retries,
            base_delay=0.01,
            timeout=1.0,
            idempotency_required=True,
            breaker=CircuitBreaker(),
            bulkhead=Bulkhead(capacity=2),
        )

    print("=== Scenario 1: 429 -> RATE_LIMITED (包含 retry_after 退避并重试成功) ===")
    mock_429 = SequenceMockTool(sequence=[{"status": 429, "retry_after": 0.02}, 200])
    cfg1 = make_config("tool_429", max_retries=1)
    registry.register(cfg1, mock_429)

    runtime = ToolRuntime(registry)
    res1 = runtime.execute("tool_429", {})
    print(f"Status: {res1['status']} | Attempts: {res1['attempts']}\n")

    print("=== Scenario 2: 503 -> RETRYABLE (识别为可重试错误，次轮成功) ===")
    mock_503 = SequenceMockTool(sequence=[503, 200])
    cfg2 = make_config("tool_503", max_retries=1)
    registry.register(cfg2, mock_503)

    res2 = runtime.execute("tool_503", {})
    print(f"Status: {res2['status']} | Attempts: {res2['attempts']}\n")

    print("=== Scenario 3: 400 -> NON_RETRYABLE (客户端错误，禁止重试直接 FAST FAIL/FATAL) ===")
    mock_400 = SequenceMockTool(sequence=[400, 200])
    cfg3 = make_config("tool_400", max_retries=2)
    registry.register(cfg3, mock_400)

    res3 = runtime.execute("tool_400", {})
    print(f"Status: {res3['status']} | ErrorKind: {res3['error_kind']} | Attempts: {res3['attempts']} (Expected: 1)\n")

    print("=== Scenario 4: RuntimeError -> INTERNAL (捕获未处理的代码异常，归一化为不可重试内部错误) ===")
    mock_crash = SequenceMockTool(sequence=[RuntimeError("DB Connection Lost"), 200])
    cfg4 = make_config("tool_crash", max_retries=2)
    registry.register(cfg4, mock_crash)

    res4 = runtime.execute("tool_crash", {})
    print(f"Status: {res4['status']} | ErrorKind: {res4['error_kind']} | Attempts: {res4['attempts']} (Expected: 1)\n")

=== Scenario 1: 429 -> RATE_LIMITED (包含 retry_after 退避并重试成功) ===
Status: SUCCESS | Attempts: 2

=== Scenario 2: 503 -> RETRYABLE (识别为可重试错误，次轮成功) ===
Status: SUCCESS | Attempts: 2

=== Scenario 3: 400 -> NON_RETRYABLE (客户端错误，禁止重试直接 FAST FAIL/FATAL) ===
Status: FATAL_ERROR | ErrorKind: non_retryable | Attempts: 1 (Expected: 1)

=== Scenario 4: RuntimeError -> INTERNAL (捕获未处理的代码异常，归一化为不可重试内部错误) ===
Status: FATAL_ERROR | ErrorKind: internal | Attempts: 1 (Expected: 1)

